# SLM-RL — Colab bake & live stream

Pick a game, bake a high-quality teacher pack (DQN + demos for Atari), and **stream progress + episode screens** in this notebook.

**Runtime → Change runtime type → GPU (T4)** is recommended for Atari DQN.

After baking you can push the pack to Hugging Face and paste the repo into the local playground (**Teacher → dqn** + DQN HF repo).

## 1. Install

Set `REPO_URL` to your fork/clone if needed, then run the cell.

In [ ]:
# @title Clone + install SLM-RL
REPO_URL = "https://github.com/craftsmanlabs/SLM-RL.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

import os
import sys
from pathlib import Path

# Prefer an already-uploaded checkout; else clone into /content/SLM-RL.
ROOT = Path("/content/SLM-RL")
here = Path.cwd()
if (here / "pyproject.toml").is_file() and (here / "slm_rl").is_dir():
    ROOT = here
elif (here / "SLM-RL" / "pyproject.toml").is_file():
    ROOT = here / "SLM-RL"
elif not (ROOT / "pyproject.toml").is_file():
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {ROOT}
else:
    print(f"already cloned: {ROOT}")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Colab has CUDA — use torch from pip + atari. cpu-train conflicts with cuda extras.
import subprocess

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("-e", ".[atari]")
pip("torch", "ipywidgets", "matplotlib", "huggingface_hub")

import torch
print("cwd:", Path.cwd())
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 2. Select game & bake knobs

In [ ]:
# @title Controls
import ipywidgets as W
from IPython.display import display

from slm_rl.games.registry import available_games
from slm_rl.packs import ATARI_GAMES, is_atari

GAMES = available_games()
ATARI = sorted(g for g in GAMES if is_atari(g))
TEXT = sorted(g for g in GAMES if g not in ATARI)
# Flat list — Atari first (Colab GPU path), then text games.
OPTIONS = ATARI + TEXT

game_dd = W.Dropdown(
    options=OPTIONS,
    value="boxing" if "boxing" in OPTIONS else OPTIONS[0],
    description="Game",
    style={"description_width": "initial"},
    layout=W.Layout(width="360px"),
)
episodes = W.IntSlider(value=200, min=10, max=2000, step=10, description="Demo episodes")
dqn_decisions = W.IntSlider(value=10_000, min=0, max=200_000, step=1000, description="DQN decisions")
seed = W.IntText(value=0, description="Seed")
stream_every = W.IntSlider(value=25, min=1, max=200, step=1, description="Stream every N steps")
hf_repo = W.Text(
    value="",
    placeholder="org/slm-rl-boxing (optional push)",
    description="HF push",
    layout=W.Layout(width="420px"),
)
hf_token = W.Password(value="", placeholder="hf_… write token if pushing", description="HF token")

ui = W.VBox([
    W.HTML("<b>Bake settings</b> — Atari trains DQN then rolls demos; text games skip DQN."),
    game_dd, episodes, dqn_decisions, seed, stream_every, hf_repo, hf_token,
])
display(ui)

def selected_game() -> str:
    return str(game_dd.value)


## 3. Bake with live stream

Runs DQN (Atari) then teacher demos. The right panel updates with the **game screen** (Atari) or **text observation** (board games) while demos roll.

In [ ]:
# @title Run bake (streaming)
from __future__ import annotations

import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display
import ipywidgets as W

from slm_rl.config.loader import load_game_config
from slm_rl.datagen.writer import RolloutWriter
from slm_rl.games.registry import get_game
from slm_rl.packs import is_atari, packs_root, push_pack, write_manifest
from slm_rl.rollout.runner import EpisodeRunner
from slm_rl.teachers import make_teacher

HOME = Path("/content/slm-rl-runs")
HOME.mkdir(parents=True, exist_ok=True)

log_out = W.Output(layout=W.Layout(height="280px", overflow="auto", border="1px solid #444"))
frame_out = W.Output(layout=W.Layout(height="280px", border="1px solid #444"))
status = W.HTML(value="<i>Idle</i>")
display(W.HBox([W.VBox([W.HTML("<b>Log</b>"), log_out]), W.VBox([W.HTML("<b>Live view</b>"), frame_out])]))
display(status)


def _log(msg: str) -> None:
    with log_out:
        print(msg, flush=True)


def _show_text(text: str, title: str) -> None:
    with frame_out:
        clear_output(wait=True)
        print(title)
        print(text[:2000])


def _show_rgb(rgb: np.ndarray, title: str) -> None:
    with frame_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(4.5, 3.2))
        ax.imshow(rgb)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
        display(fig)
        plt.close(fig)


def _ale_rgb(game) -> np.ndarray | None:
    """RGB screen from an ALE RAM env (same Stella state)."""
    env = getattr(game, "_env", None)
    if env is None:
        return None
    try:
        return env.unwrapped.ale.getScreenRGB()
    except Exception:
        return None


def bake_with_stream(
    game: str,
    *,
    episodes: int,
    dqn_decisions: int,
    seed: int,
    stream_every: int,
    push: str | None,
    token: str | None,
) -> Path:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    pack_dir = packs_root(HOME) / game
    pack_dir.mkdir(parents=True, exist_ok=True)
    rollouts_dir = pack_dir / "rollouts"
    if rollouts_dir.exists():
        shutil.rmtree(rollouts_dir)
    rollouts_dir.mkdir(parents=True)

    game_cfg = load_game_config(game)
    dqn_path = None
    if is_atari(game) and dqn_decisions > 0:
        from slm_rl.teachers.dqn import train_dqn

        dqn_path = pack_dir / "dqn.pt"
        status.value = f"<b>Training DQN</b> on {game} ({dqn_decisions} decisions, {device})…"
        _log(f"[bake] {game}: train-dqn decisions={dqn_decisions} device={device}")
        train_dqn(game_cfg, decisions=dqn_decisions, out_path=dqn_path, device=device, seed=seed)
        _log(f"[bake] wrote {dqn_path}")

    agent, model_id = make_teacher(
        game_cfg, seed=seed, dqn_checkpoint=str(dqn_path) if dqn_path else None,
    )
    game_cls = get_game(game)
    out_jsonl = rollouts_dir / f"{game}.jsonl"
    wins = 0
    status.value = f"<b>Rolling demos</b> ({episodes} episodes, {model_id})…"
    _log(f"[bake] {game}: teacher demos episodes={episodes} ({model_id})")

    with RolloutWriter(out_jsonl) as writer:
        for i in range(episodes):
            g = game_cls(game_cfg)
            # Stream frames by wrapping step: EpisodeRunner owns the loop,
            # so we run one episode then peek ALE RGB / last text from a
            # short live probe on the first steps of each streamed episode.
            runner = EpisodeRunner(
                g, agent, game_cfg, writer=writer,
                run_id=f"bake-{game}", generation=0, model_id=model_id,
            )
            # Instrument: monkeypatch step to update the live panel.
            orig_step = g.step
            step_i = {"n": 0}

            def step_and_stream(action):
                # EpisodeRunner passes an ActionSpec (not a raw id string).
                result = orig_step(action)
                step_i["n"] += 1
                if step_i["n"] % max(1, stream_every) == 0:
                    aid = getattr(action, "id", action)
                    title = f"{game}  ep {i + 1}/{episodes}  step {step_i['n']}  action={aid}"
                    rgb = _ale_rgb(g)
                    if rgb is not None:
                        _show_rgb(rgb, title)
                    else:
                        _show_text(result.observation.text, title)
                return result

            g.step = step_and_stream  # type: ignore[method-assign]
            summary = runner.run_episode(seed + i, episode_id=f"bake-{seed + i}")
            wins += summary["outcome"] == "win"
            if (i + 1) % 10 == 0 or i + 1 == episodes:
                _log(f"[bake] {game}: {i + 1}/{episodes} episodes  last_outcome={summary.get('outcome')}")

    write_manifest(pack_dir, game=game, n_episodes=episodes, has_dqn=dqn_path is not None)
    _log(f"[bake] {game}: win_rate={wins}/{episodes} → {pack_dir}")
    status.value = f"<b>Done</b> — pack at <code>{pack_dir}</code>"

    if push:
        status.value = f"<b>Pushing</b> to Hugging Face <code>{push}</code>…"
        url = push_pack(pack_dir, push, token=token or None)
        _log(f"[bake] pushed {push}: {url}")
        status.value = f"<b>Pushed</b> — <a href='{url}' target='_blank'>{push}</a>"
    return pack_dir


game = selected_game()
pack = bake_with_stream(
    game,
    episodes=int(episodes.value),
    dqn_decisions=int(dqn_decisions.value),
    seed=int(seed.value),
    stream_every=int(stream_every.value),
    push=(hf_repo.value.strip() or None),
    token=(hf_token.value.strip() or None),
)
print("PACK_DIR =", pack)

## 4. Replay stream (watch a finished episode)

After bake, stream PNG-style frames from the JSONL the same way the playground **Watch screen** does.

In [ ]:
# @title Replay one episode from the pack
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import clear_output, display

from slm_rl.config.loader import load_game_config
from slm_rl.packs import is_atari

PACK = Path("/content/slm-rl-runs/packs") / selected_game()
jsonl = next((PACK / "rollouts").glob("*.jsonl"))

# Collect episode ids (first 20)
episodes_found: list[str] = []
with jsonl.open(encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        eid = rec.get("episode_id")
        if eid and eid not in episodes_found:
            episodes_found.append(eid)
        if len(episodes_found) >= 20:
            break

print("episodes:", episodes_found[:10], "…" if len(episodes_found) > 10 else "")
EPISODE_ID = episodes_found[0]  # @param {type:"string"}
FPS = 15  # @param {type:"number"}

if not is_atari(selected_game()):
    print("Text game — printing observation stream instead of RGB.")
    with jsonl.open(encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec.get("episode_id") != EPISODE_ID:
                continue
            clear_output(wait=True)
            print(f"step {rec.get('step_idx')} action={rec.get('parsed_action')}")
            msgs = rec.get("prompt_messages") or []
            # Last user message usually carries the rendered board / RAM text.
            text = ""
            for m in reversed(msgs):
                if m.get("role") == "user" and m.get("content"):
                    text = m["content"]
                    break
            print(text or json.dumps({k: rec.get(k) for k in ("parsed_action", "reward", "outcome")}, indent=2))
            time.sleep(1 / max(FPS, 1))
            if rec.get("terminated") or rec.get("truncated"):
                break
else:
    import ale_py
    import gymnasium as gym

    gym.register_envs(ale_py)
    cfg = load_game_config(selected_game())
    env_id = cfg.extra["env_id"]
    action_repeat = int(cfg.extra.get("action_repeat", 3))
    env = None
    action_ids = None
    try:
        with jsonl.open(encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                if rec.get("episode_id") != EPISODE_ID:
                    continue
                if env is None:
                    env = gym.make(
                        env_id,
                        obs_type="ram",
                        frameskip=4,
                        repeat_action_probability=0.0,
                        render_mode="rgb_array",
                    )
                    env.reset(seed=int(rec["seed"]))
                    action_ids = list(env.unwrapped.get_action_meanings())
                action = rec.get("parsed_action")
                if action not in action_ids:
                    break
                aid = action_ids.index(action)
                for _ in range(action_repeat):
                    _, _, term, trunc, _ = env.step(aid)
                    frame = env.render()
                    clear_output(wait=True)
                    fig, ax = plt.subplots(figsize=(5, 3.5))
                    ax.imshow(frame)
                    ax.set_title(f"{EPISODE_ID}  {action}")
                    ax.axis("off")
                    display(fig)
                    plt.close(fig)
                    time.sleep(1 / max(FPS, 1))
                    if term or trunc:
                        break
                if rec.get("terminated") or rec.get("truncated"):
                    break
    finally:
        if env is not None:
            env.close()
print("replay done")

## 5. Download pack / use in local playground

- Pack path: `/content/slm-rl-runs/packs/<game>/` (`dqn.pt` + `rollouts/` + `MANIFEST.json`)
- If you pushed to HF, paste that repo into the Vue app when creating a project with **Teacher = dqn**.
- Or zip-download the pack below and copy `dqn.pt` into your host `runs/packs/<game>/`.

In [ ]:
# @title Zip pack for download
import shutil
from pathlib import Path
from google.colab import files

game = selected_game()
pack = Path("/content/slm-rl-runs/packs") / game
assert pack.is_dir(), f"missing pack {pack} — run bake first"
zip_path = Path("/content") / f"slm-rl-pack-{game}"
archive = shutil.make_archive(str(zip_path), "zip", root_dir=pack)
print("downloading", archive)
files.download(archive)